## 1. Setup and Artefact Load.
Prepare the training environment and load the unified dataset artefacts created during `01_data_eda.ipynb`.
#### Rationale:
- Establish a **reproducible** experimental context (fixed random seed and deterministic cuDNN - NVIDIA CUDA Deep Neural Network library).
- Resolve canonical project paths and verify existence of required directories (`data`, `experiments/pools/`, etc.).
- Load the **pooled table** (`source_pool_rel.csv`) and the **preprocessing configuration** (`preprocessing_config.json`) that define the 3-class problem (`Normal`, `Pneumonia`, `TB`).
- Create the output directories for checkpoints and metrics to keep training runs auditable.
- Print a concise summary confirming *GPU availability*, *ImageNet normalisation*, and *reproducibility* settings.

In [1]:
from pathlib import Path
import os, json, random, numpy as np, pandas as pd, torch

# 1. Reproducibility.
#   Use a fixed seed across all random libraries (Python, NumPy, and PyTorch)
# to make training results numerically repeatable on this workstation and future reruns.
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Disable non-deterministic kernels.
    torch.backends.cudnn.benchmark = False # Prevent cuDNN autotune drift.

SEED = 42
set_seed(SEED)

# 2. Resolve canonical project paths.
#   This notebook lives in notebooks/, therefore ".." points to the repository root.
PROJECT_ROOT    = Path("..").resolve()
CHEXPERT_PATH   = PROJECT_ROOT / "data" / "chexpert"
TBX11K_PATH     = PROJECT_ROOT / "data" / "tbx11k"
POOLS_DIR       = PROJECT_ROOT / "experiments" / "pools"
CHECKPOINTS_DIR = PROJECT_ROOT / "experiments" / "checkpoints"
RESULTS_DIR     = PROJECT_ROOT / "experiments" / "results"

# 3. Load persisted artefacts from 01_data_EDA.
#   The pooled CSV contains relative paths, class labels, and source tags.
# The JSON holds preprocessing constants (IMG_SIZE, ImageNet mean/std, label map, transform mode).
pool_csv = POOLS_DIR / "source_pool_rel.csv"
cfg_path = POOLS_DIR / "preprocessing_config.json"

if not pool_csv.exists() or not cfg_path.exists():
    raise FileNotFoundError("Missing pooled artefacts. Please run 01_data_eda.ipynb first "
                            "to generate experiments/pools/source_pool_rel.csv and preprocessing_config.json."
    )
source_pool_rel = pd.read_csv(pool_csv)
cfg = json.loads(cfg_path.read_text())

CLASS_TO_INDEX = {k: int(v) for k, v in cfg["class_to_index"].items()}
INDEX_TO_CLASS = {v: k for k, v in CLASS_TO_INDEX.items()}
IMG_SIZE       = int(cfg["img_size"])
IMAGENET_MEAN  = cfg["imagenet_mean"]
IMAGENET_STD   = cfg["imagenet_std"]
TRANSFORM_MODE = cfg.get("transform_mode", "center-crop")

# 5. Device and precision information.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(device.type == "cuda") # Enable mixed precision if a GPU is present.

# 6. Compact readiness log.
print("03_model_training - Environment")
print(f'- Device: {device.type.upper()} | PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
if device.type == "cuda":
    print(f'- GPU: {torch.cuda.get_device_name(0)} | cuDNN: {torch.backends.cudnn.version()}')

print("\nInputs (from 01/02):")
print(f'- Pool rows: {len(source_pool_rel):,}')
print(f'- Class map: {CLASS_TO_INDEX}')
print(f'- Image size: {IMG_SIZE} | Transform mode: {TRANSFORM_MODE}')
print(f'- ImageNet mean/std: {IMAGENET_MEAN} / {IMAGENET_STD}')

print("\nOutputs:")
print(f'- Checkpoints: {CHECKPOINTS_DIR.relative_to(PROJECT_ROOT)}')
print(f'- Metrics: {RESULTS_DIR.relative_to(PROJECT_ROOT)}')

print("\nReproducibility:")
print(f'- Seed: {SEED} | Deterministic cuDNN: {torch.backends.cudnn.deterministic}')
print(f'- Precision: {"Mixed (AMP)" if USE_AMP else "FP32"}')

03_model_training - Environment
- Device: CUDA | PyTorch: 2.10.0.dev20250930+cu128 | CUDA: 12.8
- GPU: NVIDIA GeForce RTX 5090 Laptop GPU | cuDNN: 91002

Inputs (from 01/02):
- Pool rows: 13,651
- Class map: {'Pneumonia': 0, 'TB': 1, 'Normal': 2}
- Image size: 256 | Transform mode: letterbox
- ImageNet mean/std: [0.485, 0.456, 0.406] / [0.229, 0.224, 0.225]

Outputs:
- Checkpoints: experiments/checkpoints
- Metrics: experiments/results

Reproducibility:
- Seed: 42 | Deterministic cuDNN: True
- Precision: Mixed (AMP)


## 2. Module Import Path Setup.
Ensure that the project's internal modules stored under '/src' (such as `/src/dataset.py`,`/src/transforms.py`, and `/src/loaders.py`) are discoverable when the notebook is executed from within the `/notebooks` directory.
By explicitely adding the parent directory to `sys.path`, we make sure that Python recongnises the custom package structure regardless of runtime environment (local workstation, remote server, or container).
#### Rationale:
- Avoids import errors when running notebooks from nested directories by dynamically adding the project root to the Python path.
- Guarantees **modular reproducibility** and **environmental portability** across local and cloud-based executions.
- Aligns the notebook workflow with the structure used for later **containerised and Kubernetes deployments**.

In [2]:
import sys
from pathlib import Path

# Ensure the project root (one level up from 'notebooks/') is visible to Python
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to sys.path:", PROJECT_ROOT)

Project root added to sys.path: /lab/px/msc-sept-24-ft-cohort-capstone-chrisanich


## 3. Shared Preprocessing Transforms.
Construct the shared preprocessing pipelines for both **training** and **evaluation** datasets using the parameters persisted during *01_data_eda.ipynb*.
These transforms include deterministic resizing, cropping, and ImageNet-compatible normalisation, ensuring that both CheXpert and TBX11K images are processed identically.
#### Rationale:
- Enforces **standarised input statistics** across datasets for fair comparison between custom and pretrained models.
- Aligns CXR input space with the **ImageNet pretraining domain**, enabling effective transfer learning.
- Promotes **reproducibility** and **consistency** in all subsequent experiments by applying identical preprocessing logic.

In [3]:
# Shared preprocessing transforms (Train / Eval)
# Build shared preprocessing transforms using parameters from preprocessing_config.json.
# These transforms ensure consistent resizing, cropping, and ImageNet normalisation
# across both datasets (CheXpert and TBX11K).

from src.transforms import build_shared_transforms

train_transforms_shared, eval_transforms_shared = build_shared_transforms(
    img_size=IMG_SIZE,
    imagenet_mean=IMAGENET_MEAN,
    imagenet_std=IMAGENET_STD,
)

print(f'Transforms built successfully: {IMG_SIZE}×{IMG_SIZE} (ImageNet-normalised)')

Transforms built successfully: 256×256 (ImageNet-normalised)


## 4. Stratified Train/Validation DataLoaders.
Build the final **train** and **validation** DataLoaders by splitting the unified dataset into 80/20 **stratified** partition.
This ensures that all three classes, `Pneumonia`, `TB`, and `Normal`, maintain proportional representation in both subsets.
Attach the appropriate transform pipeline to each split and configure **DataLoaders** optimised for GPU throughput.
#### Rationale:
- Preserves **class balance** across splits, preventing training bias and enabling robust evaluation.
- Ensures that **data augmentation** is applied only to the **training** subset while **evaluation** remains deterministic.
- Uses **parallel workers** and **pinned memory* to maximise GPU utilisation and minimise I/O latency.
- Facilitates **reproducible, high-throughput training** under fixed seeds and controlled environment conditions.

In [4]:
from src.loaders import build_dataloaders

train_loader, val_loader = build_dataloaders(
    source_pool_rel=source_pool_rel,
    class_to_index=CLASS_TO_INDEX,
    chexpert_root=CHEXPERT_PATH,
    tbx_root=TBX11K_PATH,
    img_size=IMG_SIZE,
    train_transforms=train_transforms_shared,
    eval_transforms=eval_transforms_shared,
    seed=SEED,
    batch_size=64,
    num_workers=4,
)

## 5. Model Initialisation and Training Configuration.
Define and initialise the neural network architecture, loss function, and optimisation strategy used for model training and evaluation.
This section ensures that both **custom** and **pretrained** models follow a unified configuration.
#### Rationale:
- Maintain **consistency** across all model types (**custom CNN** and **transfer-learning backbones**).
- Enable **reproducible training** by fixing seeds and using identical optimiser and scheduler settings.
- Support **fair comparison** of model performance under controlled experimental conditions.